# Preprocessing Data: retail_store_sales.csv
## Untuk Analisis Association Rule Mining (Apriori & FP-Growth)

**Langkah-langkah Preprocessing:**
1. Import Library
2. Load Dataset
3. Eksplorasi Data Awal (EDA)
4. Data Cleaning (Missing Values, Duplikat)
5. Transformasi Data ke Format Market Basket
6. Transformasi One-Hot Encoding
7. Analisis Association Rules (Apriori)
8. Export Hasil Preprocessing

## 1. Import Library

In [ ]:
%pip install pandas openpyxl mlxtend

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimport!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('retail_store_sales.csv')

print(f'Jumlah baris: {df.shape[0]}')
print(f'Jumlah kolom: {df.shape[1]}')
print(f'\nKolom dataset:')
print(df.columns.tolist())
df.head(10)

## 3. Eksplorasi Data Awal (EDA)

In [ ]:
print('=== Info Dataset ===')
df.info()

In [ ]:
print('=== Statistik Deskriptif ===')
df.describe()

In [ ]:
# Cek missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct.round(2)})
print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Cek duplikat
print('=== Duplikat ===')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

In [ ]:
# Nilai unik kolom kategorikal
print('=== Nilai Unik ===')
print(f"Jumlah Transaction ID unik: {df['Transaction ID'].nunique()}")
print(f"Jumlah Customer ID unik: {df['Customer ID'].nunique()}")
print(f"\nKategori produk ({df['Category'].nunique()} kategori):")
print(df['Category'].value_counts())
print(f"\nJumlah Item unik: {df['Item'].nunique()}")
print(f"\nPayment Method:")
print(df['Payment Method'].value_counts())
print(f"\nLocation:")
print(df['Location'].value_counts())

## 4. Data Cleaning

In [ ]:
print(f'Ukuran dataset awal: {df.shape}')

# 4.1 Hapus baris duplikat
df_clean = df.drop_duplicates()
print(f'Setelah hapus duplikat: {df_clean.shape}')

# 4.2 Hapus baris dengan missing value pada kolom Item
print(f"\nMissing pada 'Item': {df_clean['Item'].isnull().sum()}")
df_clean = df_clean.dropna(subset=['Item'])
print(f"Setelah hapus missing 'Item': {df_clean.shape}")

# 4.3 Hapus baris dengan missing value pada Quantity & Total Spent
df_clean = df_clean.dropna(subset=['Quantity', 'Total Spent'])
print(f"Setelah hapus missing 'Quantity' & 'Total Spent': {df_clean.shape}")

print(f'\n=== Hasil Cleaning ===')
print(f'Dataset bersih: {df_clean.shape}')
print(f'Data dihapus: {df.shape[0] - df_clean.shape[0]} baris')

In [ ]:
# Verifikasi data bersih
print('=== Verifikasi Data Bersih ===')
print(df_clean.isnull().sum())
print(f'\nJumlah duplikat: {df_clean.duplicated().sum()}')
df_clean.head(10)

## 5. Transformasi ke Format Market Basket

Mengelompokkan item berdasarkan **Customer ID + Bulan Transaksi** untuk membentuk keranjang belanja.
Setiap baris = daftar item unik yang dibeli seorang customer dalam 1 bulan.

In [ ]:
# Buat kolom periode bulanan
df_clean = df_clean.copy()
df_clean['YearMonth'] = pd.to_datetime(df_clean['Transaction Date']).dt.to_period('M')

# Grouping item unik per Customer + Bulan
transactions = df_clean.groupby(['Customer ID', 'YearMonth'])['Item'].apply(lambda x: sorted(set(x))).reset_index()
transactions.columns = ['Customer ID', 'YearMonth', 'Items']
transactions['Item(s)'] = transactions['Items'].apply(len)

print(f'Jumlah keranjang: {len(transactions)}')
print(f'Rata-rata item/keranjang: {transactions["Item(s)"].mean():.2f}')
print(f'Max item/keranjang: {transactions["Item(s)"].max()}')
print(f'Min item/keranjang: {transactions["Item(s)"].min()}')
print(f'\nDistribusi jumlah item per keranjang:')
print(transactions['Item(s)'].value_counts().sort_index())

transactions.head()

In [ ]:
# Pecah list items menjadi kolom: Item(s), Item 1, Item 2, ...
max_items = transactions['Item(s)'].max()
item_columns = [f'Item {i+1}' for i in range(max_items)]

items_expanded = transactions['Items'].apply(pd.Series)
items_expanded.columns = item_columns[:items_expanded.shape[1]]

df_datasets = pd.concat([transactions[['Item(s)']], items_expanded], axis=1)

print(f'Shape: {df_datasets.shape}')
print(f'Kolom: Item(s), Item 1, Item 2, ..., Item {max_items}')
df_datasets.head(10)

## 6. Transformasi One-Hot Encoding

Mengubah data transaksi menjadi format one-hot encoding (0/1):
- Setiap kolom = 1 item unik
- Setiap baris = 1 keranjang belanja
- Nilai 1 = item dibeli, 0 = tidak dibeli

In [ ]:
# Gunakan TransactionEncoder dari mlxtend
te = TransactionEncoder()
te_array = te.fit(transactions['Items']).transform(transactions['Items'])
df_onehot = pd.DataFrame(te_array.astype(int), columns=te.columns_)

print(f'Shape one-hot: {df_onehot.shape}')
print(f'Jumlah item unik: {df_onehot.shape[1]}')
df_onehot.head()

## 7. Analisis Association Rules (Apriori)

Menjalankan algoritma Apriori untuk menemukan frequent itemsets dan menghasilkan association rules.

In [ ]:
# Apriori - frequent itemsets
df_onehot_bool = df_onehot.astype(bool)
frequent_itemsets = apriori(df_onehot_bool, min_support=0.1, use_colnames=True, max_len=2, low_memory=True)

print(f'Jumlah frequent itemsets: {len(frequent_itemsets)}')
frequent_itemsets.sort_values('support', ascending=False).head(15)

In [ ]:
# Generate Association Rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5, num_itemsets=len(frequent_itemsets))

# Konversi kolom numerik untuk menghindari TypeError
numeric_cols = ['support', 'confidence', 'lift', 'conviction', 'leverage',
                'antecedent support', 'consequent support']
for col in numeric_cols:
    if col in rules.columns:
        rules[col] = pd.to_numeric(rules[col], errors='coerce')

df_rules = pd.DataFrame({
    'Premises': rules['antecedents'].apply(lambda x: ', '.join(list(x))),
    'Premises Frequency': rules['antecedent support'].apply(lambda x: round(x * len(df_onehot))),
    'Conclusion': rules['consequents'].apply(lambda x: ', '.join(list(x))),
    'Conclusion Frequency': rules['consequent support'].apply(lambda x: round(x * len(df_onehot))),
    'Premise and Conclusion Frequency': rules['support'].apply(lambda x: round(x * len(df_onehot))),
    'Support': rules['support'].values,
    'Confidence': rules['confidence'].values,
    'Lift': rules['lift'].values,
    'Conviction': rules['conviction'].values,
    'Leverage': rules['leverage'].values
})

# Round numeric columns
for col in ['Support', 'Confidence', 'Lift', 'Conviction', 'Leverage']:
    df_rules[col] = pd.to_numeric(df_rules[col], errors='coerce').round(6)

df_rules = df_rules.sort_values('Confidence', ascending=False).reset_index(drop=True)

print(f'Jumlah Association Rules: {len(df_rules)}')
df_rules.head(15)

## 8. Export Hasil Preprocessing

In [ ]:
output_file = 'retail_store_sales_preprocessed.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_datasets.to_excel(writer, sheet_name='Datasets', index=False)
    df_onehot.to_excel(writer, sheet_name='Tranform to one-hot', index=False)
    df_rules.to_excel(writer, sheet_name='Association Rules to ExampleSet', index=False)

print(f'✅ File berhasil disimpan: {output_file}')
print(f'\n   Sheet 1: Datasets                        ({df_datasets.shape[0]} baris x {df_datasets.shape[1]} kolom)')
print(f'   Sheet 2: Tranform to one-hot              ({df_onehot.shape[0]} baris x {df_onehot.shape[1]} kolom)')
print(f'   Sheet 3: Association Rules to ExampleSet   ({df_rules.shape[0]} baris x {df_rules.shape[1]} kolom)')

In [ ]:
# Ringkasan
print('=' * 60)
print('RINGKASAN PREPROCESSING')
print('=' * 60)
print(f'Dataset awal            : {df.shape[0]} baris x {df.shape[1]} kolom')
print(f'Setelah cleaning        : {df_clean.shape[0]} baris x {df_clean.shape[1]} kolom')
print(f'Data dihapus            : {df.shape[0] - df_clean.shape[0]} baris')
print(f'Jumlah keranjang        : {len(df_datasets)}')
print(f'Jumlah item unik        : {df_onehot.shape[1]}')
print(f'Max item/keranjang      : {df_datasets["Item(s)"].max()}')
print(f'Rata-rata item/keranjang: {df_datasets["Item(s)"].mean():.2f}')
print(f'Jumlah Association Rules : {len(df_rules)}')
print(f'Output                  : {output_file}')
print('=' * 60)